# Kahneman Framing RCT × TRIBE v2 (text only)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akifnu/DSprojects/blob/main/tribev2/notebooks/Framing_RCT_NoSetup.ipynb)

**Pure text RCT** — 318 gain/loss pairs (636 texts), 63,600 trial assignments in the dataset.

1. Runtime → **A100 GPU** (40 GB+; text needs LLaMA 3.2)
2. Accept [LLaMA 3.2 license](https://huggingface.co/meta-llama/Llama-3.2-3B) on Hugging Face
3. Runtime → **Run all** (twice if runtime restarts after install)

When prompted, paste a Hugging Face **read token** once (or add Colab secret `HF_TOKEN`).

In [ ]:
NOTEBOOK_VERSION = 'text-rct-2026-06-16'
print('Notebook', NOTEBOOK_VERSION, '| text-only | 318 scenario pairs')

In [ ]:
import subprocess, sys
from pathlib import Path

MARKER = Path('/content/.tribev2_text_deps')

if not MARKER.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'numpy'], check=False)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy==2.2.6'])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'tribev2 @ git+https://github.com/facebookresearch/TRIBEv2.git',
        'huggingface_hub', 'scipy', 'pandas', 'scikit-learn',
    ])
    MARKER.write_text('ok')
    print('Installed numpy==2.2.6 + tribev2. Restarting once — then Run all again.')
    import IPython
    IPython.get_ipython().kernel.do_shutdown(restart=True)
else:
    import numpy as np
    from tribev2 import TribeModel
    assert np.__version__ == '2.2.6'
    print('numpy', np.__version__, '| tribev2 OK')

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login

if not os.environ.get('HF_TOKEN'):
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        os.environ['HF_TOKEN'] = getpass('HuggingFace read token (LLaMA 3.2 access): ')

os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '600'
os.environ['HF_HUB_HTTP_TIMEOUT'] = '600'
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('Hugging Face login OK')

In [ ]:
import json, urllib.request

BASE = 'https://raw.githubusercontent.com/akifnu/DSprojects/main/tribev2/data/framing_rct'
scenarios_doc = json.loads(urllib.request.urlopen(f'{BASE}/scenarios.json').read().decode())
protocol = json.loads(urllib.request.urlopen(f'{BASE}/protocol.json').read().decode())
SCENARIOS = scenarios_doc['scenarios']

print('RCT dataset loaded from GitHub')
print('  scenario pairs:', len(SCENARIOS))
print('  unique texts:', len(SCENARIOS) * 2)
print('  subjects:', protocol['n_subjects'])
print('  trial assignments:', protocol['n_trials'])
print('  example gain:', SCENARIOS[0]['gain_frame'][:80], '...')

In [ ]:
# How many pairs to run this session (318 = full study; start with 12 on T4)
MAX_SCENARIOS = 12
RUN = SCENARIOS[:MAX_SCENARIOS]
print(f'Running text inference on {len(RUN)} / {len(SCENARIOS)} pairs')

In [ ]:
import numpy as np
from tribev2 import TribeModel

model = TribeModel.from_pretrained('facebook/tribev2', cache_folder='/content/tribe_cache', device='cuda')
print('TRIBE v2 loaded')

In [ ]:
import os, tempfile

def predict_text(text: str) -> np.ndarray:
    tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8')
    try:
        tmp.write(text.strip())
        tmp.flush()
        os.fsync(tmp.fileno())
        tmp.close()
        events = model.get_events_dataframe(text_path=tmp.name)
        preds, _ = model.predict(events=events, verbose=False)
        return np.asarray(preds)
    finally:
        if os.path.exists(tmp.name):
            os.unlink(tmp.name)

results = []
for s in RUN:
    row = {'id': s['scenario_id'], 'domain': s['domain']}
    for frame, key in [('gain', 'gain_frame'), ('loss', 'loss_frame')]:
        preds = predict_text(s[key])
        row[f'{frame}_mean_abs'] = float(np.mean(np.abs(preds)))
    row['loss_minus_gain'] = row['loss_mean_abs'] - row['gain_mean_abs']
    results.append(row)
    print(s['scenario_id'], f"{row['loss_minus_gain']:+.4f}")

print('finished', len(results), 'text pairs')

In [ ]:
import pandas as pd
from scipy import stats

df = pd.DataFrame(results)
display(df[['id', 'domain', 'gain_mean_abs', 'loss_mean_abs', 'loss_minus_gain']])

diff = df['loss_mean_abs'].values - df['gain_mean_abs'].values
_, p = stats.ttest_rel(df['loss_mean_abs'], df['gain_mean_abs'])
print(f"\nRan {len(df)} text pairs from {len(SCENARIOS)}-pair RCT")
print(f"loss>gain: {(diff>0).sum()}/{len(df)}  mean_diff={diff.mean():.4f}  p={p:.4f}")
print('Increase MAX_SCENARIOS toward 318 for the full massive RCT run.')